<a href="https://colab.research.google.com/github/Sopitas24/CPSN/blob/main/Simulacion_de_examen_javiera_cortes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

07 Preparación Examen Unidad - Simulación del Examen (Core).txt
100%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

np.random.seed(42)

data = pd.DataFrame({
    "user_id": range(1, 301),
    "app_version": np.random.choice(["1.0", "1.1", "1.2", "1.3"], 300),
    "platform": np.random.choice(["Android", "iOS"], 300),
    "session_duration": np.random.randint(1, 180, 300),
    "number_of_sessions": np.random.randint(1, 20, 300),
    "country": np.random.choice(
        ["USA", "Canada", "Mexico", "UK", "Germany", "France", "Spain", "Italy"],
        300
    ),
    "user_feedback": np.random.randint(1, 6, 300)
})

data.to_csv("user_app_data.csv", index=False)


datos = pd.read_csv(
    "user_app_data.csv",
    dtype={
        "user_id": "string",
        "app_version": "string",
        "platform": "string",
        "country": "string"
    }
)

print("PRIMERAS 10 FILAS")
display(datos.head(10))

print("\nINFORMACIÓN ORIGINAL")
datos.info()

print("\nESTADÍSTICAS ORIGINALES")
display(datos.describe())

filas_originales = len(datos)

print("\nPLATAFORMAS ORIGINALES")
print(datos["platform"].value_counts(dropna=False))

print("\nPAÍSES ORIGINALES")
print(datos["country"].value_counts(dropna=False))

columnas_texto = ["user_id", "app_version", "platform", "country"]

for columna in columnas_texto:
    datos[columna] = datos[columna].astype("string").str.strip()
    datos[columna] = datos[columna].replace("", pd.NA)

plataformas = {
    "android": "Android",
    "ios": "iOS"
}

datos["platform"] = datos["platform"].str.lower().map(plataformas)

paises = {
    "usa": "USA",
    "us": "USA",
    "united states": "USA",
    "estados unidos": "USA",
    "canada": "Canada",
    "canadá": "Canada",
    "mexico": "Mexico",
    "méxico": "Mexico",
    "uk": "UK",
    "united kingdom": "UK",
    "reino unido": "UK",
    "germany": "Germany",
    "alemania": "Germany",
    "france": "France",
    "francia": "France",
    "spain": "Spain",
    "españa": "Spain",
    "italy": "Italy",
    "italia": "Italy"
}

paises_limpios = datos["country"].str.lower().map(paises)
datos["country"] = paises_limpios.fillna(datos["country"])



columnas_numericas = [
    "session_duration",
    "number_of_sessions",
    "user_feedback"
]

for columna in columnas_numericas:
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce")
    datos[columna] = datos[columna].replace(
        [np.inf, -np.inf],
        np.nan
    )

print("\nVALORES FALTANTES TRAS CONVERTIR LOS DATOS")
print(datos.isna().sum())

duplicados = datos.duplicated().sum()
datos = datos.drop_duplicates().copy()

print("\nFilas completamente duplicadas eliminadas:", duplicados)

usuarios_repetidos = (
    datos["user_id"].notna()
    & datos["user_id"].duplicated(keep=False)
)

conflictos = datos.loc[usuarios_repetidos].copy()
conflictos.to_csv("usuarios_para_revision.csv", index=False)

datos = datos.loc[~usuarios_repetidos].copy()

print("Filas con identificadores en conflicto apartadas:", len(conflictos))

duracion_invalida = (
    (datos["session_duration"] < 0)
    | (datos["session_duration"] > 1440)
)

sesiones_invalidas = (
    (datos["number_of_sessions"] < 0)
    | (datos["number_of_sessions"] % 1 != 0)
)

feedback_invalido = (
    (datos["user_feedback"] < 1)
    | (datos["user_feedback"] > 5)
    | (datos["user_feedback"] % 1 != 0)
)

print("\nVALORES INAPROPIADOS")
print("Duraciones inválidas:", duracion_invalida.sum())
print("Cantidades de sesiones inválidas:", sesiones_invalidas.sum())
print("Puntuaciones inválidas:", feedback_invalido.sum())


datos.loc[duracion_invalida, "session_duration"] = np.nan
datos.loc[sesiones_invalidas, "number_of_sessions"] = np.nan
datos.loc[feedback_invalido, "user_feedback"] = np.nan

columnas_esenciales = [
    "user_id",
    "session_duration",
    "number_of_sessions"
]

filas_antes = len(datos)
datos = datos.dropna(subset=columnas_esenciales).copy()

print(
    "\nFilas eliminadas por datos esenciales faltantes:",
    filas_antes - len(datos)
)

if datos.empty:
    raise ValueError("No quedaron registros válidos para analizar.")

for columna in ["platform", "country", "app_version"]:
    datos[columna] = datos[columna].fillna("Sin información")


print("\nMÉTODOS UTILIZADOS PARA LOS FALTANTES")
print("- Identificador o uso faltante: eliminación, para no inventar registros.")
print("- Categorías faltantes: etiqueta 'Sin información'.")
print("- Feedback faltante: se conserva y se excluye de los cálculos de puntuación.")

datos["user_id"] = datos["user_id"].astype("string")
datos["app_version"] = datos["app_version"].astype("string")
datos["platform"] = datos["platform"].astype("category")
datos["country"] = datos["country"].astype("category")
datos["session_duration"] = datos["session_duration"].astype(float)
datos["number_of_sessions"] = datos["number_of_sessions"].astype("Int64")
datos["user_feedback"] = datos["user_feedback"].astype("Int64")

datos = datos.reset_index(drop=True)

print("\nTIPOS DE DATOS FINALES")
print(datos.dtypes)

print("\nRESUMEN DE LIMPIEZA")
print("Filas originales:", filas_originales)
print("Filas disponibles:", len(datos))
print("Identificadores únicos:", datos["user_id"].nunique())

print("\nVALORES FALTANTES FINALES")
print(datos.isna().sum())


plt.figure(figsize=(8, 4))
plt.hist(
    datos["session_duration"],
    bins=15,
    color="steelblue",
    edgecolor="white"
)
plt.title("Distribución de la duración de las sesiones")
plt.xlabel("Duración de la sesión (minutos)")
plt.ylabel("Cantidad de usuarios")
plt.tight_layout()
plt.show()


print("\nINTERPRETACIÓN DEL HISTOGRAMA")
print(
    f"La duración mediana es "
    f"{datos['session_duration'].median():.1f} minutos."
)
print(
    f"Las duraciones van de {datos['session_duration'].min():.1f} "
    f"a {datos['session_duration'].max():.1f} minutos."
)
print("El histograma muestra en qué intervalos se concentran las sesiones.")


conteo_plataformas = datos["platform"].value_counts()

plt.figure(figsize=(7, 4))
plt.bar(
    conteo_plataformas.index.astype(str),
    conteo_plataformas.values,
    color="seagreen"
)
plt.title("Cantidad de usuarios por plataforma")
plt.xlabel("Plataforma")
plt.ylabel("Cantidad de usuarios")
plt.tight_layout()
plt.show()

print("\nINTERPRETACIÓN DEL GRÁFICO DE BARRAS")

mayor_frecuencia = conteo_plataformas.max()
plataformas_mayores = conteo_plataformas[
    conteo_plataformas == mayor_frecuencia
].index.astype(str).tolist()

print(
    "Categoría o categorías más frecuentes:",
    ", ".join(plataformas_mayores)
)
print(f"Cada una tiene {mayor_frecuencia} usuarios en los datos analizados.")
print("La frecuencia indica presencia en la muestra, no satisfacción ni retención.")

plt.figure(figsize=(8, 4))
plt.scatter(
    datos["number_of_sessions"].astype(float),
    datos["session_duration"],
    alpha=0.5,
    color="darkorange"
)
plt.title("Número de sesiones y duración de la sesión")
plt.xlabel("Número de sesiones en un día")
plt.ylabel("Duración de la sesión (minutos)")
plt.tight_layout()
plt.show()

correlacion = datos["number_of_sessions"].astype(float).corr(
    datos["session_duration"]
)

print("\nINTERPRETACIÓN DEL GRÁFICO DE DISPERSIÓN")

if pd.isna(correlacion):
    print("No se puede calcular la correlación: falta variación o suficientes datos.")
else:
    print(f"Correlación lineal de Pearson: {correlacion:.2f}.")
    print(
        "Un valor cercano a 0 indica poca relación lineal; "
        "cercano a 1 o -1 indica una relación lineal más marcada."
    )

print("Una asociación entre las variables no demuestra una relación de causa y efecto.")

datos_feedback = datos.dropna(subset=["user_feedback"]).copy()
datos_feedback = datos_feedback[
    datos_feedback["platform"].isin(["Android", "iOS"])
]

print("\nINTERPRETACIÓN DEL DIAGRAMA DE CAJAS")

if datos_feedback.empty:
    print("No hay puntuaciones válidas con plataforma conocida para comparar.")
else:
    plataformas = [
        plataforma
        for plataforma in ["Android", "iOS"]
        if (datos_feedback["platform"] == plataforma).any()
    ]

    grupos = [
        datos_feedback.loc[
            datos_feedback["platform"] == plataforma,
            "user_feedback"
        ].astype(float).to_numpy()
        for plataforma in plataformas
    ]

    plt.figure(figsize=(7, 4))
    plt.boxplot(grupos)
    plt.xticks(range(1, len(plataformas) + 1), plataformas)
    plt.title("Puntuación de experiencia por plataforma")
    plt.xlabel("Plataforma")
    plt.ylabel("Puntuación del usuario")
    plt.yticks([1, 2, 3, 4, 5])
    plt.tight_layout()
    plt.show()

    resumen_feedback = datos_feedback.groupby(
        "platform",
        observed=True
    )["user_feedback"].agg(["count", "mean", "median"])

    display(resumen_feedback.round(2))

    print("La línea dentro de cada caja representa la mediana.")
    print("La caja muestra el 50 % central de las puntuaciones.")
    print("Como las puntuaciones solo van de 1 a 5, puede haber muchos empates.")
    print("Las diferencias observadas no prueban que una plataforma sea mejor.")

print("\nCONCLUSIONES")
print("Se revisaron duplicados, faltantes, categorías, tipos y valores inválidos.")
print("Se exploraron duración, plataforma, frecuencia de uso y experiencia.")
print("Los datos fueron generados al azar y sirven para practicar.")
print(
    "No podemos medir retención con este archivo: "
    "necesitamos observar a los mismos usuarios en distintas fechas."
)

datos.to_csv("user_app_data_limpio.csv", index=False)

print("\nARCHIVOS DISPONIBLES EN EL PANEL DE COLAB")
print("- user_app_data.csv: datos originales.")
print("- user_app_data_limpio.csv: datos preparados para el análisis.")
print("- usuarios_para_revision.csv: identificadores en conflicto, si los hubo.")
Mostrando 07 Preparación Examen Unidad - Simulación del Examen (Core).txt